# Historical mask-generation scratchpad

This preserved notebook has a multi-lesion limitation: `get_ellipse_from_bbox` assigns the current label to every positive pixel after drawing each ellipse, so later lesions relabel earlier foreground. This can overwrite earlier categories on images with lesions of different types. Its final visualization also references a commented-out `mask` variable. This is source evidence, not a validated Run All recipe. Original code below is unchanged; outputs and data are not included.


In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd

In [ ]:
def get_ellipse_from_bbox(mask, x, y, w, h, label=1):
    mask = cv2.ellipse(mask.copy(), (int(x + w / 2), int(y + h / 2)), (int(w / 2), int(h / 2)), 0, 0, 360, 255, -1)
    mask[mask > 0] = label
    return mask

In [ ]:
ROOT = "/mnt/stor/datasets/CBIS-DDSM/"

In [ ]:
df = pd.read_csv(os.path.join(ROOT, "cleaned_cbis_ddsm.csv"))
df["label"] = df.pathology.apply(lambda x: x.lower().split("_")[0]) + "_" + df.abnormality.apply(lambda x: x.lower())
print(df.label.value_counts())
df.head()

In [ ]:
label_dict = {k: v + 1 for v, k in enumerate(sorted(df.label.unique()))}
label_dict

In [ ]:
save_dir = os.path.join(ROOT, "masks")
os.makedirs(save_dir, exist_ok=True)

In [ ]:
for filename, file_df in df.groupby("filepath"):
    img = cv2.imread(os.path.join(ROOT, "png", filename), cv2.IMREAD_GRAYSCALE)
    # mask = np.zeros_like(img)
    mask_ellipse = np.zeros_like(img)
    for index, row in file_df.iterrows():
        x1, y1, x2, y2 = row[["y1", "x1", "y2", "x2"]]
        # mask[y1:y2, x1:x2] = label_dict[row.label]
        mask_ellipse = get_ellipse_from_bbox(mask_ellipse, x1, y1, x2 - x1, y2 - y1, label_dict[row.label])
    print(np.unique(mask_ellipse))
    cv2.imwrite(os.path.join(save_dir, filename), mask_ellipse)

In [ ]:
plt.subplot(1, 3, 1)
plt.imshow(img, cmap="gray")
plt.subplot(1, 3, 2)
plt.imshow(mask, cmap="gray")
plt.subplot(1, 3, 3)
plt.imshow(mask_ellipse, cmap="gray")
plt.show()

In [ ]:
np.unique(mask_ellipse)